In [0]:
# Install required libraries
%pip install mlflow scipy numpy pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import numpy as np
import pandas as pd
from scipy import stats
import mlflow
import mlflow.sklearn
from datetime import datetime, timedelta

print("Libraries imported successfully!")

Libraries imported successfully!


In [0]:
np.random.seed(42)
n_users = 1000

# Pre-experiment features (30 days before experiment)
pre_conversion_rate = np.random.beta(2, 8, n_users)  # historical conversion rate
pre_revenue = np.random.exponential(50, n_users)      # historical revenue

# Experiment assignment
treatment = np.random.binomial(1, 0.5, n_users)

# Experiment outcomes (treatment has ~10% lift)
base_conversion = pre_conversion_rate * 0.8
converted = np.random.binomial(1, base_conversion + treatment * 0.02, n_users)
revenue = pre_revenue * 0.9 + treatment * 5 + np.random.normal(0, 10, n_users)

# Build dataframe
df = pd.DataFrame({
    'user_id': range(n_users),
    'treatment': treatment,
    'pre_conversion_rate': pre_conversion_rate,
    'pre_revenue': pre_revenue,
    'converted': converted,
    'revenue': revenue
})

print(f"Dataset created: {len(df)} users")
print(f"Control: {(df.treatment==0).sum()} | Treatment: {(df.treatment==1).sum()}")
print(f"Overall conversion rate: {df.converted.mean():.3f}")
print(df.head())

Dataset created: 1000 users
Control: 497 | Treatment: 503
Overall conversion rate: 0.170
   user_id  treatment  pre_conversion_rate  pre_revenue  converted     revenue
0        0          1             0.247183    56.826083          0   41.349033
1        1          0             0.164191   134.221862          0  153.201067
2        2          1             0.317524   289.621135          1  274.611074
3        3          0             0.108755    63.676410          0   63.338410
4        4          1             0.363810    16.929208          0   28.188156


In [0]:
def apply_cuped(df, outcome_col, covariate_col):
    y = df[outcome_col].values
    x = df[covariate_col].values
    
    # Calculate theta (regression coefficient)
    theta = np.cov(y, x)[0, 1] / np.var(x)
    
    # Apply CUPED adjustment
    df = df.copy()
    df['outcome_cuped'] = y - theta * (x - x.mean())
    
    # Calculate variance reduction
    var_original = np.var(y)
    var_cuped = np.var(df['outcome_cuped'])
    var_reduction = (1 - var_cuped / var_original) * 100
    
    print(f"Theta: {theta:.4f}")
    print(f"Original variance: {var_original:.6f}")
    print(f"CUPED variance: {var_cuped:.6f}")
    print(f"Variance reduction: {var_reduction:.1f}%")
    
    return df, theta, var_reduction

# Apply CUPED on conversion rate
df, theta, var_reduction = apply_cuped(df, 'converted', 'pre_conversion_rate')
print("\nCUPED applied successfully!")

Theta: 0.7953
Original variance: 0.141100
CUPED variance: 0.132348
Variance reduction: 6.2%

CUPED applied successfully!


In [0]:
def bayesian_ab_test(df, outcome_col='outcome_cuped'):
    control = df[df['treatment'] == 0][outcome_col].values
    treatment = df[df['treatment'] == 1][outcome_col].values
    
    # Beta posteriors
    prior_alpha, prior_beta = 1, 1
    
    control_posterior = stats.beta(
        prior_alpha + int(control.sum()),
        prior_beta + len(control) - int(control.sum())
    )
    treatment_posterior = stats.beta(
        prior_alpha + int(treatment.sum()),
        prior_beta + len(treatment) - int(treatment.sum())
    )
    
    # Monte Carlo sampling
    n_samples = 100_000
    control_samples = control_posterior.rvs(n_samples)
    treatment_samples = treatment_posterior.rvs(n_samples)
    
    prob_treatment_better = (treatment_samples > control_samples).mean()
    expected_lift = (treatment_samples - control_samples).mean()
    ci = np.percentile(treatment_samples - control_samples, [2.5, 97.5])
    
    print(f"P(Treatment > Control): {prob_treatment_better:.3f}")
    print(f"Expected Lift: {expected_lift:.4f}")
    print(f"95% Credible Interval: [{ci[0]:.4f}, {ci[1]:.4f}]")
    
    decision = "SHIP" if prob_treatment_better > 0.95 else "HOLD - need more data"
    print(f"\nDecision: {decision}")
    
    return prob_treatment_better, expected_lift, ci

prob, lift, ci = bayesian_ab_test(df)

P(Treatment > Control): 0.991
Expected Lift: 0.0557
95% Credible Interval: [0.0097, 0.1024]

Decision: SHIP


In [0]:
mlflow.set_experiment("/Users/shloksheth.13@gmail.com/experi_cuped_bayesian")

with mlflow.start_run(run_name="checkout_cta_test") as run:
    # Log parameters
    mlflow.log_param("metric", "conversion_rate")
    mlflow.log_param("variance_method", "CUPED")
    mlflow.log_param("inference_method", "Bayesian")
    mlflow.log_param("covariate", "pre_conversion_rate")
    mlflow.log_param("covariate_window_days", 30)
    mlflow.log_param("n_users", len(df))
    mlflow.log_param("start_date", "2026-05-01")
    mlflow.log_param("end_date", "2026-05-14")
    
    # Log metrics
    mlflow.log_metric("cuped_theta", theta)
    mlflow.log_metric("variance_reduction_pct", var_reduction)
    mlflow.log_metric("prob_treatment_better", prob)
    mlflow.log_metric("expected_lift", lift)
    mlflow.log_metric("ci_lower", ci[0])
    mlflow.log_metric("ci_upper", ci[1])
    
    run_id = run.info.run_id
    print(f"MLflow run logged successfully!")
    print(f"Run ID: {run_id}")
    print(f"Decision: {'SHIP' if prob > 0.95 else 'HOLD'}")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


MLflow run logged successfully!
Run ID: ce89b30524e24c09b74175fff82ccff9
Decision: SHIP


In [0]:
# Save results to Delta table
results = spark.createDataFrame([{
    'experiment_id': run_id,
    'experiment_name': 'checkout_cta_test',
    'prob_treatment_better': float(prob),
    'expected_lift': float(lift),
    'variance_reduction_pct': float(var_reduction),
    'decision': 'SHIP' if prob > 0.95 else 'HOLD',
    'run_date': '2026-06-05'
}])

results.write.format("delta") \
    .mode("append") \
    .saveAsTable("experi_ab_test_results")

print("Results saved to Delta table!")
spark.sql("SELECT * FROM experi_ab_test_results").show()

Results saved to Delta table!
+--------+-------------------+--------------------+-----------------+---------------------+----------+----------------------+
|decision|      expected_lift|       experiment_id|  experiment_name|prob_treatment_better|  run_date|variance_reduction_pct|
+--------+-------------------+--------------------+-----------------+---------------------+----------+----------------------+
|    SHIP|0.05573111771600735|4c615f46c3ac498a8...|checkout_cta_test|              0.99056|2026-06-05|     6.202866262966666|
|    SHIP|0.05573111771600735|ce89b30524e24c09b...|checkout_cta_test|              0.99056|2026-06-05|     6.202866262966666|
+--------+-------------------+--------------------+-----------------+---------------------+----------+----------------------+



In [0]:
# Let's first create a summary function we can expose via the app
def run_experiment(n_users=1000, treatment_effect=0.02, covariate_window=30):
    
    np.random.seed(42)
    
    # Generate data
    pre_conversion_rate = np.random.beta(2, 8, n_users)
    pre_revenue = np.random.exponential(50, n_users)
    treatment = np.random.binomial(1, 0.5, n_users)
    base_conversion = pre_conversion_rate * 0.8
    converted = np.random.binomial(1, base_conversion + treatment * treatment_effect, n_users)
    
    df = pd.DataFrame({
        'user_id': range(n_users),
        'treatment': treatment,
        'pre_conversion_rate': pre_conversion_rate,
        'converted': converted
    })
    
    # CUPED
    df, theta, var_reduction = apply_cuped(df, 'converted', 'pre_conversion_rate')
    
    # Bayesian
    prob, lift, ci = bayesian_ab_test(df)
    
    return {
        'n_users': n_users,
        'var_reduction': round(var_reduction, 2),
        'prob_treatment_better': round(float(prob), 3),
        'expected_lift': round(float(lift), 4),
        'ci_lower': round(float(ci[0]), 4),
        'ci_upper': round(float(ci[1]), 4),
        'decision': 'SHIP' if prob > 0.95 else 'HOLD'
    }

# Test it
result = run_experiment(n_users=1000, treatment_effect=0.02)
print(result)

Theta: 0.7953
Original variance: 0.141100
CUPED variance: 0.132348
Variance reduction: 6.2%
P(Treatment > Control): 0.991
Expected Lift: 0.0557
95% Credible Interval: [0.0097, 0.1024]

Decision: SHIP
{'n_users': 1000, 'var_reduction': np.float64(6.2), 'prob_treatment_better': 0.991, 'expected_lift': 0.0557, 'ci_lower': 0.0097, 'ci_upper': 0.1024, 'decision': 'SHIP'}


In [0]:
# Create the Databricks App file
app_code = '''
import gradio as gr
import numpy as np
import pandas as pd
from scipy import stats

def apply_cuped(df, outcome_col, covariate_col):
    y = df[outcome_col].values
    x = df[covariate_col].values
    theta = np.cov(y, x)[0, 1] / np.var(x)
    df = df.copy()
    df["outcome_cuped"] = y - theta * (x - x.mean())
    var_reduction = (1 - np.var(df["outcome_cuped"]) / np.var(y)) * 100
    return df, theta, var_reduction

def bayesian_ab_test(df, outcome_col="outcome_cuped"):
    control = df[df["treatment"] == 0][outcome_col].values
    treatment = df[df["treatment"] == 1][outcome_col].values
    prior_alpha, prior_beta = 1, 1
    control_posterior = stats.beta(
        prior_alpha + int(control.sum()),
        prior_beta + len(control) - int(control.sum())
    )
    treatment_posterior = stats.beta(
        prior_alpha + int(treatment.sum()),
        prior_beta + len(treatment) - int(treatment.sum())
    )
    n_samples = 100_000
    control_samples = control_posterior.rvs(n_samples)
    treatment_samples = treatment_posterior.rvs(n_samples)
    prob = (treatment_samples > control_samples).mean()
    lift = (treatment_samples - control_samples).mean()
    ci = np.percentile(treatment_samples - control_samples, [2.5, 97.5])
    return prob, lift, ci

def run_analysis(n_users, treatment_effect):
    np.random.seed(42)
    n_users = int(n_users)
    pre_conversion_rate = np.random.beta(2, 8, n_users)
    pre_revenue = np.random.exponential(50, n_users)
    treatment = np.random.binomial(1, 0.5, n_users)
    base_conversion = pre_conversion_rate * 0.8
    converted = np.random.binomial(
        1, base_conversion + treatment * treatment_effect, n_users
    )
    df = pd.DataFrame({
        "user_id": range(n_users),
        "treatment": treatment,
        "pre_conversion_rate": pre_conversion_rate,
        "converted": converted
    })
    df, theta, var_reduction = apply_cuped(df, "converted", "pre_conversion_rate")
    prob, lift, ci = bayesian_ab_test(df)
    decision = "✅ SHIP IT" if prob > 0.95 else "⏳ HOLD - Need More Data"
    return (
        f"{prob:.1%}",
        f"{lift:.4f}",
        f"[{ci[0]:.4f}, {ci[1]:.4f}]",
        f"{var_reduction:.1f}%",
        decision
    )

with gr.Blocks(title="Experi - Statistical Experimentation Tool") as demo:
    gr.Markdown("# 🧪 Experi")
    gr.Markdown("### Production-grade A/B testing with CUPED + Bayesian Inference on Databricks")
    
    with gr.Row():
        n_users = gr.Slider(100, 10000, value=1000, step=100, label="Sample Size")
        treatment_effect = gr.Slider(0.0, 0.1, value=0.02, step=0.005, label="Treatment Effect")
    
    run_btn = gr.Button("Run Experiment", variant="primary")
    
    with gr.Row():
        prob_out = gr.Textbox(label="P(Treatment > Control)")
        lift_out = gr.Textbox(label="Expected Lift")
        ci_out = gr.Textbox(label="95% Credible Interval")
        var_out = gr.Textbox(label="Variance Reduction (CUPED)")
        decision_out = gr.Textbox(label="Decision")
    
    run_btn.click(
        run_analysis,
        inputs=[n_users, treatment_effect],
        outputs=[prob_out, lift_out, ci_out, var_out, decision_out]
    )

demo.launch()
'''

# Write app.py
with open('/tmp/app.py', 'w') as f:
    f.write(app_code)

print("app.py created successfully!")

app.py created successfully!


In [0]:
%pip install gradio

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import gradio as gr
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

gr.close_all()

def analyze(df):
    y = df['converted'].values
    x = df['pre_conversion_rate'].values
    theta = np.cov(y, x)[0, 1] / np.var(x)
    df['outcome_cuped'] = y - theta * (x - x.mean())
    var_reduction = (1 - np.var(df['outcome_cuped']) / np.var(y)) * 100
    
    control = df[df['treatment'] == 0]['outcome_cuped'].values
    treat = df[df['treatment'] == 1]['outcome_cuped'].values
    
    control_posterior = stats.beta(1 + int(control.sum()), 1 + len(control) - int(control.sum()))
    treat_posterior = stats.beta(1 + int(treat.sum()), 1 + len(treat) - int(treat.sum()))
    
    n_samples = 100_000
    control_samples = control_posterior.rvs(n_samples)
    treat_samples = treat_posterior.rvs(n_samples)
    
    prob = (treat_samples > control_samples).mean()
    lift = (treat_samples - control_samples).mean()
    ci = np.percentile(treat_samples - control_samples, [2.5, 97.5])
    decision = "✅ SHIP IT" if prob > 0.95 else "⏳ HOLD - Need More Data"
    
    # Plot posterior distributions
    fig, ax = plt.subplots(figsize=(10, 5))
    x_range = np.linspace(0, 0.5, 1000)
    ax.plot(x_range, control_posterior.pdf(x_range), 
            label='Control', color='#4C72B0', linewidth=2.5)
    ax.plot(x_range, treat_posterior.pdf(x_range), 
            label='Treatment', color='#DD8452', linewidth=2.5)
    ax.fill_between(x_range, control_posterior.pdf(x_range), alpha=0.2, color='#4C72B0')
    ax.fill_between(x_range, treat_posterior.pdf(x_range), alpha=0.2, color='#DD8452')
    ax.axvline(control_posterior.mean(), color='#4C72B0', linestyle='--', alpha=0.7)
    ax.axvline(treat_posterior.mean(), color='#DD8452', linestyle='--', alpha=0.7)
    ax.set_xlabel('Conversion Rate', fontsize=12)
    ax.set_ylabel('Probability Density', fontsize=12)
    ax.set_title(f'Posterior Distributions — P(Treatment > Control): {prob:.1%}', fontsize=14)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return (
        f"{prob:.1%}",
        f"{lift:.4f}",
        f"[{ci[0]:.4f}, {ci[1]:.4f}]",
        f"{var_reduction:.1f}%",
        decision,
        fig
    )

def run_synthetic(n_users, treatment_effect):
    np.random.seed(42)
    n_users = int(n_users)
    pre_conversion_rate = np.random.beta(2, 8, n_users)
    treatment = np.random.binomial(1, 0.5, n_users)
    base_conversion = pre_conversion_rate * 0.8
    converted = np.random.binomial(1, base_conversion + treatment * treatment_effect, n_users)
    df = pd.DataFrame({
        'user_id': range(n_users),
        'treatment': treatment,
        'pre_conversion_rate': pre_conversion_rate,
        'converted': converted
    })
    return analyze(df)

def run_csv(file):
    if file is None:
        return "No file uploaded", "", "", "", "", None
    try:
        df = pd.read_csv(file.name)
        for col in ['treatment', 'pre_conversion_rate', 'converted']:
            if col not in df.columns:
                return f"Missing column: {col}", "", "", "", "", None
        return analyze(df)
    except Exception as e:
        return f"Error: {str(e)}", "", "", "", "", None

def calc_sample_size(baseline_rate, mde, power, alpha):
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    p1 = baseline_rate
    p2 = baseline_rate * (1 + mde / 100)
    pooled = (p1 + p2) / 2
    n = (z_alpha * np.sqrt(2 * pooled * (1 - pooled)) +
         z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2 / (p2 - p1) ** 2
    n = int(np.ceil(n))
    total = n * 2
    return f"{n:,} per variant", f"{total:,} total", f"{p2:.3f}", f"~{total // 1000}K users needed"

with gr.Blocks(title="Experi") as demo:
    gr.Markdown("# 🧪 Experi")
    gr.Markdown("### Production-grade A/B testing with CUPED + Bayesian Inference on Databricks")
    gr.Markdown("*Helping data-driven teams make faster, more confident decisions.*")
    
    with gr.Tabs():
        with gr.Tab("🎲 Synthetic Data"):
            with gr.Row():
                n_users = gr.Slider(100, 10000, value=1000, step=100, label="Sample Size")
                treatment_effect = gr.Slider(0.0, 0.1, value=0.08, step=0.005, label="Treatment Effect")
            btn1 = gr.Button("Run Experiment", variant="primary")
            plot1 = gr.Plot(label="Posterior Distributions")
            with gr.Row():
                o1 = gr.Textbox(label="P(Treatment > Control)")
                o2 = gr.Textbox(label="Expected Lift")
                o3 = gr.Textbox(label="95% Credible Interval")
                o4 = gr.Textbox(label="Variance Reduction (CUPED)")
                o5 = gr.Textbox(label="Decision")
            btn1.click(run_synthetic, inputs=[n_users, treatment_effect], 
                      outputs=[o1, o2, o3, o4, o5, plot1])

        with gr.Tab("📁 Upload CSV"):
            gr.Markdown("**Required columns:** `treatment` (0/1), `pre_conversion_rate` (float), `converted` (0/1)")
            csv_file = gr.File(label="Upload CSV", file_types=[".csv"])
            btn2 = gr.Button("Run Experiment", variant="primary")
            plot2 = gr.Plot(label="Posterior Distributions")
            with gr.Row():
                c1 = gr.Textbox(label="P(Treatment > Control)")
                c2 = gr.Textbox(label="Expected Lift")
                c3 = gr.Textbox(label="95% Credible Interval")
                c4 = gr.Textbox(label="Variance Reduction (CUPED)")
                c5 = gr.Textbox(label="Decision")
            btn2.click(run_csv, inputs=[csv_file], 
                      outputs=[c1, c2, c3, c4, c5, plot2])

        with gr.Tab("📐 Sample Size Calculator"):
            gr.Markdown("**How many users do I need for my experiment?**")
            with gr.Row():
                baseline_rate = gr.Slider(0.01, 0.5, value=0.1, step=0.01, label="Baseline Conversion Rate")
                mde = gr.Slider(1, 50, value=10, step=1, label="Minimum Detectable Effect (%)")
            with gr.Row():
                power = gr.Slider(0.5, 0.99, value=0.8, step=0.05, label="Statistical Power")
                alpha = gr.Slider(0.01, 0.1, value=0.05, step=0.01, label="Significance Level (α)")
            btn3 = gr.Button("Calculate", variant="primary")
            with gr.Row():
                s1 = gr.Textbox(label="Per Variant")
                s2 = gr.Textbox(label="Total Users")
                s3 = gr.Textbox(label="Target Rate")
                s4 = gr.Textbox(label="Summary")
            btn3.click(calc_sample_size, inputs=[baseline_rate, mde, power, alpha], 
                      outputs=[s1, s2, s3, s4])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://162301bba8ef7109ea.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [0]:
import gradio as gr
gr.close_all()

Closing server running on port: 7870
Closing server running on port: 7865
Closing server running on port: 7866
Closing server running on port: 7863
Closing server running on port: 7864
Closing server running on port: 7868
Closing server running on port: 8080
Closing server running on port: 7869
Closing server running on port: 7867
Closing server running on port: 7860
Closing server running on port: 7862
Closing server running on port: 7861
